# YOLOv13-S Baseline - AFB / Tuberculosis6208 (Chen split)

Mirror konfigurasi dari `yolo12.ipynb` supaya **apples-to-apples** vs YOLOv12 baseline.

**Setup:**
- Dataset: 1265 image+XML (Tuberculosis6208 phone-camera).
- Split: Chen et al. IJAI 2024 - 1024/140/101, `SPLIT_SEED=42` deterministic.
- Training: SGD, lr0=0.01, cos_lr, 60 epoch, batch=16, imgsz=640, `SEED=42`.
- Output: best.pt + test eval (mAP50, mAP50-95, mAP@0.9, precision, recall).

## 1. Imports & paths

In [ ]:
import os, sys, gc, random, time
from pathlib import Path
import numpy as np
import torch

# Konfigurasi - sesuaikan dengan environment lo
REPO_DIR     = Path('D:/Project/afb-yolo13')
YOLOV13_DIR  = Path('D:/Project/yolov13')  # YOLOv13 fork (sudah pip install -e)
DATASET_SRC  = Path('D:/project/yolov12/Tuberculosis6208/tuberculosis-phonecamera')
SPLIT_DIR    = Path('D:/datasets/tb_chen_split')
RUN_PROJECT  = Path('D:/runs/afb_yolov13')
WEIGHTS_DIR  = REPO_DIR / 'weights'

DATA_YAML    = SPLIT_DIR / 'data.yaml'

MODEL        = 'yolov13s'
SEED         = 42
EPOCHS       = 60
IMGSZ        = 640
BATCH        = 16
DEVICE       = 0

RUN_NAME     = f'{MODEL}_seed{SEED}_{EPOCHS}ep'
print('cfg     :', MODEL)
print('seed    :', SEED)
print('epochs  :', EPOCHS)
print('run_name:', RUN_NAME)
print('split   :', SPLIT_DIR)
print('data    :', DATA_YAML)

## 2. Build Chen split (skip jika sudah ada)

In [ ]:
if not DATA_YAML.exists():
    cmd = (
        f'python "{REPO_DIR}/scripts/build_chen_split.py" '
        f'--src "{DATASET_SRC}" --out "{SPLIT_DIR}" --seed 42'
    )
    print(cmd)
    os.system(cmd)
else:
    print(f'Split already exists at {SPLIT_DIR}')

print('\n--- data.yaml ---')
print(DATA_YAML.read_text())

## 3. Seed + SDP kernel stability

In [ ]:
os.environ['PYTORCH_SDP_KERNEL'] = 'math'
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('Seeded.')

## 4. Auto-download pretrained yolov13s.pt

In [ ]:
from urllib.request import urlretrieve
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
url = f'https://github.com/iMoonLab/yolov13/releases/download/yolov13/{MODEL}.pt'
pt = WEIGHTS_DIR / f'{MODEL}.pt'
if not pt.exists():
    print(f'Downloading {url} -> {pt}')
    urlretrieve(url, pt)
print(f'Pretrained: {pt}  ({pt.stat().st_size/1e6:.1f} MB)')

## 5. Train - `model.train()` eksplisit, mirror yolo12.ipynb hyperparams

In [ ]:
from ultralytics import YOLO

# Build dari YAML (scale dari nama: yolov13s.yaml -> 's').
model = YOLO(f'{MODEL}.yaml')
try:
    model.load(str(pt))
    print(f'Loaded pretrained: {pt}')
except Exception as e:
    print(f'[warn] could not load pretrained: {e}')

t0 = time.time()
results = model.train(
    data=str(DATA_YAML),
    freeze=2,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    optimizer='SGD',
    lr0=0.01, lrf=0.01,
    momentum=0.937, weight_decay=0.0005,
    cos_lr=True,
    close_mosaic=10,
    hsv_h=0.1, hsv_s=0.3, hsv_v=0.3,
    degrees=30, translate=0.05, scale=0.1,
    flipud=0.3,
    mosaic=0.2, mixup=0.2,
    patience=0,
    amp=True, deterministic=True, seed=SEED, workers=8,
    project=str(RUN_PROJECT),
    name=f'{RUN_NAME}_train',
    exist_ok=True, save=True, verbose=True,
)
train_secs = time.time() - t0
print(f'\nTrain time: {train_secs/60:.1f} min')
print(f'Save dir  : {results.save_dir}')

## 6. Test eval di holdout 101-image split

In [ ]:
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best ckpt:', best_pt)

eval_model = YOLO(str(best_pt))
eva = eval_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, device=DEVICE, verbose=False)

map50   = float(eva.box.map50)
map5095 = float(eva.box.map)
precision = float(np.mean(np.atleast_1d(eva.box.p)))
recall    = float(np.mean(np.atleast_1d(eva.box.r)))

map_at_09 = float('nan')
try:
    ap_all = eva.box.all_ap
    if ap_all is not None and len(ap_all):
        ap = ap_all.mean(axis=0) if (hasattr(ap_all, 'ndim') and ap_all.ndim == 2) else ap_all
        if len(ap) >= 9: map_at_09 = float(ap[8])
except Exception as e:
    print(f'  (mAP@0.9 extract failed: {e})')

print(f'\n=== TEST RESULTS ({RUN_NAME}) ===')
print(f'  mAP50     : {map50:.4f}')
print(f'  mAP50-95  : {map5095:.4f}')
print(f'  mAP@0.9   : {map_at_09:.4f}')
print(f'  precision : {precision:.4f}')
print(f'  recall    : {recall:.4f}')
print(f'  train_min : {train_secs/60:.1f}')

## 7. Diagnose baseline -> decide arsitektur improvement

In [ ]:
DIAG_OUT = REPO_DIR / 'diag_outputs' / f'{RUN_NAME}_val'
cmd = (
    f'python "{REPO_DIR}/scripts/diagnose_baseline.py" '
    f'--ckpt "{best_pt}" '
    f'--data "{DATA_YAML}" '
    f'--split val --imgsz {IMGSZ} '
    f'--out "{DIAG_OUT}"'
)
print(cmd)
os.system(cmd)